<!--
# 13 Compuled and interpreted languages
-->

# 13 编译型和解释型语言

<!--
## Introduction

*This notebook is optional and intended for those with an interest in going beyond the course material.*

Programming languages and commonly classed as *compiled* or *interpreted*. We summarise and demonstrate some of the differences in this notebook.

### Compiled languages

A compiled language uses a *compiler* to transform input code into a program (machine code) that is executed by a computer. Machine code is the set of instructions for a computer to execute in the CPUs computers 'native' language (instruction set). It is not human readable. The compiler generally processes the entire program, transforming it in a sequence of steps into machine code.

Common and popular compiled languages include C, C++, Fortran and Rust.


### Interpreted languages

An interpreted language processes program instructions as they are encountered (line-by-line) rather processing the entire program into machine code ahead of time. Python is an interpreted language.


### Difference between compiled and interpreted languages

Compiled languages lead to programs are generally faster than interpreted programs, although in many cases implementations in an interpreted language are nowadays fast enough for the application needs. Compiled programs can have a smaller memory footprint, which can be important for embedded devices and other platforms with limited capacity. The computer on which a compiled program runs does not need to have a compiler or an interpreter installed.

When a compiler translates code into an executable program it will typically perform checks and perform optimisations (static analysis). The compiler checks for valid syntax, and sophisticated optimisations can perform code transformations that make programs faster. Interpreted languages are usually simpler to develop, and more interactive and avoid the need for a compilation step. Interpreted languages are often dynamically typed, with the interpreter inferring the types, e.g. integers versus floats, at runtime, With compiled languages, types are usually fixed at compile time.


### Just-in-time compilation

The difference between interpreted and compiled languages is not as clear as it once was, with interpreted languages now often using 'just-in-time' compilation. We will explore the impact of compiled code using [Numba](https://numba.pydata.org/), a just-in-time compiler for Python that targets numerical computation. For functions that we specifically annotate, Numba can compile the code and apply performance optimisations typical of compiled languages with the objective of making functions faster.


### Objectives

- Understand the difference between compiled and interpreted implementations
- Explore performance differences between interpreted and compiled implementations
- Awareness of intermediate representations and assembly code
-->

## 介绍

*本笔记本是可选的，仅供那些有兴趣深入了解课程材料之外内容的人阅读。*

编程语言通常分为*编译型*或*解释型*。我们在本笔记本中总结并演示其中的一些差异。

### 编译型语言

编译型语言使用*编译器*将输入代码转换为由计算机执行的程序（机器代码）。机器代码是计算机以CPU的“本机”语言（指令集）执行的一组指令。它是不可读的。编译器通常处理整个程序，将其按步骤转换为机器代码。

常见和流行的编译型语言包括C，C++，Fortran和Rust。


### 解释型语言

解释型语言在遇到程序指令时（逐行）处理它们，而不是提前将整个程序处理成机器代码。Python是一种解释型语言。


### 编译型语言和解释型语言的区别

编译型语言生成的程序通常比解释型程序快，尽管在许多情况下，解释型语言的实现在速度上已足够满足应用需求。编译型程序的内存占用可能较小，这对于嵌入式设备和其他容量有限的平台很重要。运行编译型程序的计算机不需要安装编译器或解释器。

当编译器将代码转换为可执行程序时，通常会执行检查和优化（静态分析）。编译器检查语法是否有效，并且复杂的优化可以执行使程序更快的代码转换。解释型语言通常更易于开发，更具交互性，并且避免了编译步骤。解释型语言通常是动态类型的，解释器在运行时推断类型，例如整数与浮点数。对于编译型语言，类型通常在编译时固定。


### 即时编译 (JIT)

解释型语言和编译型语言之间的区别不再像以前那样清晰，解释型语言现在经常使用“即时”编译。我们将使用[Numba](https://numba.pydata.org/)探索编译代码的影响，Numba是一个针对数值计算的Python即时编译器。对于我们要特别注释的函数，Numba可以编译代码并应用典型的编译型语言的性能优化，以此让函数更快。


### 目标

- 了解编译型实现和解释型实现之间的区别
- 探索解释型实现和编译型实现之间的性能差异
- 了解中间表示和汇编代码

<!--
We will later use `Numba`, so we install it now.
-->

我们稍后将使用`Numba`，所以我们现在安装它。

In [1]:
!pip install numba

<!--
## Performance of interpreted and compiled functions

In [07 Numerical computation](07%20Numerical%20computation.ipynb) we tested the performance of a simple function for computing the norm of a long vector. We consider a similar problem here: computing the dot product of a vector with itself, $\boldsymbol{x} \cdot \boldsymbol{x}$, using our own Python function and using NumPy:
-->

## 解释型函数和编译型函数的性能

在[07 Numerical computation](07%20Numerical%20computation.ipynb)中，我们测试了一个简单函数的性能，该函数用于计算长向量的范数。我们在这里考虑一个类似的问题：使用我们自己的Python函数和使用NumPy计算向量与其自身的点积 $\boldsymbol{x} \cdot \boldsymbol{x}$：

In [2]:
import numpy as np
import random

def compute_norm2(x):
    norm2 = 0.0
    for xi in x:
        norm2 += xi*xi
    return norm2

x = np.random.rand(10000000)

print("Native Python implementation")
%time n0 = compute_norm2(x)

print("\n NumPy implementation")
%time n1 = np.dot(x, x)

<!--
As expected, the NumPy code is many orders of magnitude faster. NumPy in fact uses compiled code/library for the computation, which is the reason why it is much faster than our pure Python implementation.

We now make a small change and add the 'decorator' `@numba.jit` to our function. This instructs Numba to transform our function in a compiled function/program.
-->

正如预期的那样，NumPy代码要快很多个数量级。NumPy实际上使用编译后的代码/库进行计算，这就是它比我们的纯Python实现快得多的原因。

我们要现在做一个小的改动，并在我们的函数中添加'装饰器' `@numba.jit`。这指示Numba将我们的函数转换为编译后的函数/程序。

In [3]:
import numba

@numba.jit(nopython=True,fastmath=True)
def compute_norm2(x):
    norm2 = 0.0
    for xi in x:
        norm2 += xi*xi
    return norm2

x = np.random.rand(10000000)
compute_norm2(x)

print("Numba implementation")
%time n0 = compute_norm2(x)

print("\n NumPy implementation")
%time n1 = np.dot(x, x)

<!--
Note that we call `compute_norm2` twice and time only the second call. We want to measure the raw cost of the computation and not the Numba just-in-time compilation overhead that is incurred the first time a function is processed.

The Numba version is much faster than the pure Python version. NumPy is faster again for this operation, but relative close to the Numba time. This is likely because NumPy is using a highly optimised BLAS (Basic Linear Algebra Subprograms) implementation, which is a set of machine code level functions that are highly tuned for basic linear algebra operations. 
-->

注意我们调用了`compute_norm2`两次，只对第二次调用计时。我们要测量的是计算的原始成本，而不是第一次处理函数时产生的Numba即时编译开销。

Numba版本比纯Python版本快得多。NumPy在这个操作上再次更快，但相对接近Numba的时间。这很可能是因为NumPy使用的是高度优化的BLAS（基本线性代数子程序）实现，这是一组针对基本线性代数操作高度调整的机器代码级函数。

<!--
## Sorting

We saw in [10 Algorithms](10%20Algorithms.ipynb) that our implementation of the quicksort algorithm was considerably slower than the Python built-in quicksort. Part of the performance difference could be explained by our implementation being in pure Python, with the built-in Python function being implemented in a compiled language.

We can explore the difference compilation might make to our implementation. To start, we reproduce the pure Python quicksort implementation:
-->

## 排序

我们在[10 Algorithms](10%20Algorithms.ipynb)中看到，我们的快速排序算法实现比Python内置的快速排序慢得多。性能差异的部分原因可以解释为我们的实现是纯Python的，而内置的Python函数是用编译型语言实现的。

我们可以探索编译可能对我们的实现产生的影响。首先，我们重现纯Python快速排序实现：

In [4]:
def partition_ref(A, lo, hi):
    """Partitioning function for use in quicksort"""
    pivot = A[hi]
    i = lo
    for j in range(lo,  hi):
        if A[j] <= pivot:
            A[i], A[j] = A[j], A[i]
            i += 1
    A[i], A[hi] = A[hi], A[i]
    return i

def quicksort_ref(A, lo=0, hi=None):
    """Sort A and return sorted array"""

    # Initialise data the first time function is called    
    if hi is None:
        A = A.copy()
        hi = len(A) - 1

    # Sort    
    if lo < hi:
        p = partition_ref(A, lo,  hi)
        quicksort_ref(A, lo, p - 1)
        quicksort_ref(A, p + 1, hi)
    return A

<!--
We now introduce a version annotated with a Numba decorator:
-->

我们现在介绍一个带有Numba装饰器注释的版本：

In [5]:
@numba.jit(nopython=True,fastmath=True)
def partition_jit(A, lo, hi):
    """Partitioning function for use in quicksort"""
    pivot = A[hi]
    i = lo
    for j in range(lo,  hi):
        if A[j] <= pivot:
            A[i], A[j] = A[j], A[i]
            i += 1
    A[i], A[hi] = A[hi], A[i]
    return i

@numba.jit(nopython=True,fastmath=True)
def quicksort_jit(A, lo=0, hi=-1):
    """Sort A and return sorted array"""

    # Initialise data the first time function is called    
    if hi == -1:
        A = A.copy()
        hi = len(A) - 1

    # Sort    
    if lo < hi:
        p = partition_jit(A, lo,  hi)
        quicksort_jit(A, lo, p - 1)
        quicksort_jit(A, p + 1, hi)
    return A

<!--
The last argument to `quicksort_jit` has been changed slightly so that the argument type does not change (argument types that change are problematic for a compiler as it needs to know ahead of time which types to generate machine code for).

We can now time our pure Python implementation, the Numba-compiled implementation and the built-in sort function. As before, we will call `quicksort_jit` once before timing to eliminate the cost of the just-in-time compilation.
-->

`quicksort_jit`的最后一个参数已稍作更改，以便参数类型不会更改（更改的参数类型对于编译器来说是有问题的，因为它需要提前知道要为哪些类型生成机器代码）。

我们现在可以对纯Python实现、Numba编译的实现和内置排序函数进行计时。如前所述，我们将在计时之前调用`quicksort_jit`一次，以消除即时编译的成本。

In [6]:
data = np.random.rand(1000000)

# Time the pure Python implementation
print("Pure Python implementation")
%time x = quicksort_ref(data)

# Time the Numba implementation
quicksort_jit(data)
print("Numba implementation")
%time x = quicksort_jit(data)

# Time the built-in implementation
print("Python built-in implementation")
%time x = np.sort(data, kind='quicksort')

<!--
The pure Python implementation is clearly the slowest. The Numba and built-in implementation are relatively close in time. Note that the Numba implementation is virtually a direct translation of the pure Python implementation and has not been carefully optimised. 
-->

纯Python实现显然是最慢的。Numba和内置实现在时间上相对接近。请注意，Numba实现实际上是纯Python实现的直接翻译，并未经过仔细优化。

<!--
## Intermediate representations and assembly code

A compiler translates input code into (i) an 'intermediate representation' (IR), and then into (ii) machine code. The IR is the compiler's internal representation of a program. A compiler can perform optimisations on the IR that may make a program faster and which may be specific to the CPU type. Machine code is the low instructions sent to the CPU.

With Numba we can inspect the IR and the assembly code. Assembly code is human readable code (but very low level) that maps almost one-to-one to machine code (which would be very hard to read).

Consider a very simple function that returns the sum of two integers:
-->

## 中间表示和汇编代码

编译器将输入代码转换为(i) '中间表示' (IR)，然后转换为(ii) 机器代码。IR是程序的编译器内部表示。编译器可以对IR执行优化，这可能会使程序更快，并且可能特定于CPU类型。机器代码是发送到CPU的低级指令。

使用Numba，我们可以检查IR和汇编代码。汇编代码是人类可读的代码（但非常低级），几乎一对一映射到机器代码（这很难阅读）。

考虑一个非常简单的函数，它返回两个整数的和：

In [7]:
from numba import int64

@numba.jit('int64(int64, int64)', nopython=True)
def add(x, y):
    return x + y

add(2, 3)

<!--
Not that we have specified the argument types in this case.


### Intermediate representation

We can inspect the compiler's IR for the this function:
-->

注意我们在这种情况下指定了参数类型。


### 中间表示

我们可以检查此函数的编译器IR：

In [8]:
for v, k in add.inspect_llvm().items():
    print(k)

<!--
The IR would be of interest to someone designing compilers or seeing the optimisation transformations that a compiler might perform.

### Assembly code

In some very special cases it can be helpful to inspect the assembly code, which is the closest to readable version of CPU instructions. It is usually inspected only in cases where an understanding of the lowest level operations is required, e.g. when extreme performance is necessary. It is specific to a CPU architecture.
-->

IR对于设计编译器或查看编译器可能执行的优化转换的人来说可能很有趣。

### 汇编代码

在某些非常特殊的情况下，检查汇编代码可能会有所帮助，这是最接近人类可读版本的CPU指令。通常仅在需要了解最低级别操作的情况下（例如当需要极高性能时）才检查它。它特定于CPU架构。

In [9]:
for v, k in add.inspect_asm().items():
    print(k)

<!--
## Exercises

Select exercises from the previous notebooks that could be made faster using Numba and investigate what speed-ups you can achieve.
-->

## 练习

从以前的笔记本中选择可以使用Numba加快速度的练习，并研究可以实现的加速效果。